In [2]:
# ── Encoding Dengan Pipeline ────────────────────────────────────────────
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# Dataset gabungan: numerik + kategorik
data_gabungan = {
    'usia':       [22, 35, 28, 45, 30, 52, 24, 41],
    'gaji':       [3e6, 8.5e6, 5.2e6, 12e6, 6e6, 15e6, 3.5e6, 10e6],
    'kota':       ['Jakarta','Bandung','Surabaya','Jakarta',
                   'Bandung','Surabaya','Jakarta','Bandung'],
    'churn':      [1, 0, 1, 0, 0, 0, 1, 0]
}
df = pd.DataFrame(data_gabungan)

In [3]:
X = df[['usia', 'gaji', 'kota']]
y = df['churn']  # target tidak di-encode dengan OHE — biarkan 0/1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [4]:
# Definisikan kolom mana yang dapat treatment apa
numeric_features     = ['usia', 'gaji']
categorical_features = ['kota']

# ColumnTransformer: proses kolom berbeda secara paralel
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                          numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features),
])

In [5]:
# Pipeline: preprocessing + model dalam satu objek
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(random_state=42))
])

pipeline.fit(X_train, y_train)   # fit HANYA pada training data
score = pipeline.score(X_test, y_test)
print(f"Accuracy: {score:.2f}")

# Saat predict, pipeline otomatis:
# 1. transform input baru dengan scaler yang sama
# 2. encode kategori dengan encoder yang sama
# 3. baru masuk ke model

Accuracy: 1.00


Kenapa pipeline adalah cara yang benar:

- fit hanya sekali pada training data — tidak ada kebocoran informasi ke test
- Saat predict di production, transformasi terjadi otomatis dan konsisten
- Kamu tidak bisa lupa scale satu kolom saja